# Solar Filament Post-Processing Optimization

This notebook performs a grid search over post-processing thresholds (`probability_threshold` and `watershed_seed_threshold`) using the trained weights from `experiment_014` to maximize the Mean Penalized Instance Dice score.

In [ ]:
from pathlib import Path

# Import functions from the main experiment script
import experiment_014 as exp
import matplotlib.pyplot as plt
import numpy as np
import yaml

## 1. Load Data and Config

In [ ]:
config = yaml.safe_load(exp.EMBEDDED_CONFIG_YAML)
data_root = exp.resolve_data_root(config)

# Load observations
observations, _, _ = exp.load_observation_records(data_root / config["data"]["train_annotations"])
assignments, _ = exp.assign_grouped_folds(observations, n_splits=config["validation"]["n_splits"], seed=config["validation"]["seed"])

# Isolate Fold 0 for validation
val_obs = [obs for obs in observations if assignments[obs["observation_key"]] == 0]
print(f"Loaded {len(val_obs)} validation observations for Fold 0.")

## 2. Load Model and Generate Raw Predictions
*Note: This requires your trained PyTorch weights to be saved locally.*

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = exp.build_model(config["model"]["base_channels"]).to(device)

# TODO: Point this to your saved weights from the Kaggle output
weights_path = Path("../kaggle_output_check/fold_0_best.pth")
if weights_path.exists():
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval()
    print("Weights loaded successfully.")
else:
    print(f"Warning: Weights not found at {weights_path}. Grid search requires trained weights.")

## 3. Grid Search Thresholds
We will sweep through `probability_threshold` and `watershed_seed_threshold` and evaluate the Instance Dice.

In [ ]:
import itertools

prob_thresholds = [0.35, 0.40, 0.45, 0.50]
seed_thresholds = [0.20, 0.25, 0.30, 0.35]
min_areas = [32, 48, 64, 96]

print("Starting Grid Search...")
best_score = -1
best_params = {}

# NOTE: In a real run, you should pre-calculate the model's raw probabilities
# for all validation images so you don't run the neural network inside the loop.
# For demonstration, we assume `mask_predictions` and `dist_predictions` are cached.
mask_predictions = [] # Fill with pre-calculated outputs
dist_predictions = [] # Fill with pre-calculated outputs

if mask_predictions:
    for p_thresh, s_thresh, area in itertools.product(prob_thresholds, seed_thresholds, min_areas):
        # Override config
        test_config = config.copy()
        test_config["postprocessing"]["probability_threshold"] = p_thresh
        test_config["postprocessing"]["watershed_seed_threshold"] = s_thresh
        test_config["postprocessing"]["min_component_area_at_model_resolution"] = area
        
        result = exp.evaluate_oof(val_obs, mask_predictions, dist_predictions, config=test_config)
        score = result["instance"]["mean_penalized_instance_dice"]
        
        if score > best_score:
            best_score = score
            best_params = {'prob': p_thresh, 'seed': s_thresh, 'area': area}
            
    print(f"Best Score: {best_score:.4f}")
    print(f"Best Params: {best_params}")
else:
    print("Pre-calculate predictions first to run the sweep efficiently.")

## 4. Visualize Results

In [ ]:
def plot_instances(image, instances, title="Instances"):
    plt.figure(figsize=(10, 10))
    plt.imshow(image, cmap='gray')
    # Overlay colored instances
    overlay = np.zeros((*image.shape, 4), dtype=np.float32)
    for idx, inst in enumerate(instances):
        color = plt.cm.hsv((idx + 1) / max(len(instances), 1))
        overlay[inst > 0] = color
    plt.imshow(overlay, alpha=0.5)
    plt.title(title)
    plt.axis('off')
    plt.show()

# plot_instances(val_obs[0]["image"], predicted_instances)